# Lesson 5 · Asking Business Questions: Grouping, Aggregating, Pivoting & Merging
### From rows of records to answers that a decision-maker can use

| | |
|---|---|
| **Type** | LESSON 05 |
| **Time** | about 150 minutes |
| **Level** | Intermediate |
| **Prerequisites** | Lessons 3–4 |

**By the end of this notebook you will be able to:**

- Explain *split → apply → combine* and use `groupby` with `agg`, named aggregation, `transform` and `filter`
- Build league tables, volatility measures and within-group comparisons
- Reshape with `pivot_table`, `crosstab` and `melt`
- Combine tables with `concat` and `merge` (inner/left/right/outer), and guard against silent row explosions and key mismatches
- Assemble a multi-source dataset (prices + production + weather + counties) ready for analysis

## 🛠️ Setup
Run this cell first. It imports the tools and loads the data every problem below relies on.

In [1]:

import sys; sys.path.insert(0, "..")
import json
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check, first_look

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

prices = pd.read_csv(DATA_DIR / "market_prices_clean.csv", parse_dates=["date"])
prices["year"] = prices["date"].dt.year
prices["month"] = prices["date"].dt.month

production = pd.read_excel(DATA_DIR / "production_by_county.xlsx", sheet_name="production", skiprows=2)
counties = pd.read_excel(DATA_DIR / "production_by_county.xlsx", sheet_name="counties")
weather = pd.read_json(DATA_DIR / "weather_monthly.json")
with open(DATA_DIR / "markets.json") as f:
    markets = pd.json_normalize(json.load(f))
markets.columns = markets.columns.str.replace(".", "_", regex=False)
print("Setup complete ✔  prices", prices.shape, "| production", production.shape, "| weather", weather.shape)


Setup complete ✔  prices (15048, 11) | production (204, 5) | weather (432, 5)


## 👋 From records to answers

So far we asked questions about *individual rows*. Decision-makers ask questions about **groups**:

* *"Which county pays farmers the most for tomatoes?"*
* *"Which crop's price is the most unpredictable?"*
* *"Do counties that grow more maize have cheaper maize?"*

Every one of these has the same shape: **split** the table into groups, **apply** a calculation to each group, **combine** the results.
That is the entire idea of `groupby`.

```
          SPLIT                     APPLY                 COMBINE
  ┌─────────────────┐       ┌───────────────┐
  │ crop  price     │       │ Maize  → mean │        crop      mean
  │ Maize   50      │──┐    └───────────────┘        Beans     120
  │ Beans  120      │  ├──► Maize: [50, 60]  → 55 ──►Maize      55
  │ Maize   60      │  │    Beans: [120,130] → 125
  │ Beans  130      │──┘
  └─────────────────┘
```

## 1 · `groupby` basics

In [2]:
tiny = pd.DataFrame({"crop": ["Maize", "Beans", "Maize", "Beans"], "price": [50, 120, 60, 130]})
print(tiny.groupby("crop")["price"].mean())

crop
Beans    125.0
Maize     55.0
Name: price, dtype: float64


In [3]:
by_crop = prices.groupby("crop")["price_kes_per_kg"].mean().round(1).sort_values(ascending=False)
print(by_crop)

crop
Beans       155.3
Onions       86.0
Tomatoes     77.3
Maize        63.4
Bananas      49.5
Potatoes     47.1
Kale         38.7
Cabbage      31.2
Name: price_kes_per_kg, dtype: float64


> 💡 **Intuition.** Read `prices.groupby('crop')['price_kes_per_kg'].mean()` aloud as *“for each crop, take the price column and average it.”* If you can say the sentence, you can write the code.

### 1.1 Many statistics at once: `agg`

In [4]:
prices.groupby("crop")["price_kes_per_kg"].agg(["count", "mean", "median", "std", "min", "max"]).round(1)

,count,mean,median,std,min,max
crop,,,,,,
Bananas,1672,49.5,49.3,5.8,35.4,69.2
Beans,2090,155.3,154.4,19.0,108.5,213.7
Cabbage,1881,31.2,30.9,4.5,20.5,46.6
Kale,2090,38.7,38.8,5.6,23.8,58.2
Maize,1881,63.4,62.9,11.1,38.0,91.6
Onions,1672,86.0,84.8,12.4,53.4,130.1
Potatoes,2090,47.1,47.0,7.2,29.1,70.8
Tomatoes,1672,77.3,75.7,15.1,38.9,138.1


### 1.2 Named aggregation: the readable, professional way

Different columns, different statistics, **custom output names**: `new_name=("column", "function")`.

In [5]:
summary = prices.groupby("crop").agg(
    n_records=("record_id", "count"),
    avg_price=("price_kes_per_kg", "mean"),
    max_price=("price_kes_per_kg", "max"),
    total_volume_t=("volume_kg", lambda s: s.sum() / 1000),     # a custom function (tonnes)
).round(1)
summary.sort_values("avg_price", ascending=False)

,n_records,avg_price,max_price,total_volume_t
crop,,,,
Beans,2090,155.3,213.7,32414.5
Onions,1672,86.0,130.1,26260.7
Tomatoes,1672,77.3,138.1,37744.5
Maize,1881,63.4,91.6,80153.8
Bananas,1672,49.5,69.2,31054.7
Potatoes,2090,47.1,70.8,65277.3
Kale,2090,38.7,58.2,39233.9
Cabbage,1881,31.2,46.6,44996.6


### 1.3 Several keys, and tidy output

In [6]:
cc = (prices.groupby(["county", "crop"], as_index=False)["price_kes_per_kg"].mean()
      .rename(columns={"price_kes_per_kg": "avg_price"}).round(1))
print(cc.shape, "-> one row per (county, crop) pair")
cc.head(6)

(66, 3) -> one row per (county, crop) pair


,county,crop,avg_price
0,Kiambu,Beans,164.5
1,Kiambu,Cabbage,33.6
2,Kiambu,Kale,41.0
3,Kiambu,Onions,93.6
4,Kiambu,Potatoes,50.0
5,Kiambu,Tomatoes,86.5


> 📝 **Note.** With `as_index=False` (or `.reset_index()`) the group keys stay as normal columns: easier to merge and plot later. Without it they become the **index** (a MultiIndex for several keys).

### 1.4 Group by a *calculated* key

You can group by any Series of the same length, for example, the year pulled from the date:

In [7]:
yearly = prices.groupby("year")["price_kes_per_kg"].mean().round(2)
print(yearly)
print("\nOverall price growth 2021 → 2024: %.1f%%" % ((yearly[2024] / yearly[2021] - 1) * 100))

year
2021    60.96
2022    68.55
2023    71.59
2024    74.64
Name: price_kes_per_kg, dtype: float64

Overall price growth 2021 → 2024: 22.4%


> 🧠 **Pause and predict.** Our generator built in roughly 7% annual food inflation. Does the yearly average reflect it? (Not exactly: the 2022-23 drought pushed maize/bean prices up *more* in those years, and the mix of crops matters.)

### 1.5 `size` vs `count`, and `nunique`

* `size()` counts **rows** in each group (including NaN).
* `count()` counts **non-missing** values of a column.
* `nunique()` counts **distinct** values.

In [8]:
g = prices.groupby("county")
print(pd.DataFrame({"rows": g.size(), "distinct_markets": g["market"].nunique(), "distinct_crops": g["crop"].nunique()}))

             rows  distinct_markets  distinct_crops
county                                             
Kiambu       1254                 1               6
Kisumu       1672                 1               8
Machakos     1254                 1               6
Meru         1254                 1               6
Mombasa      1672                 1               8
Nairobi      2926                 2               8
Nakuru       1672                 1               8
Nyeri        1672                 1               8
Uasin Gishu  1672                 1               8


### 1.6 A volatility measure: the coefficient of variation

*Which crop's price is the most unpredictable?* The standard deviation alone is misleading (beans cost more, so they vary more in KES).
Divide by the mean to compare fairly: **CV = std ÷ mean**.

In [9]:
g = prices.groupby("crop")["price_kes_per_kg"]
cv = (g.std() / g.mean()).sort_values(ascending=False).round(3)
print(cv)

crop
Tomatoes    0.195
Maize       0.175
Potatoes    0.152
Kale        0.145
Onions      0.144
Cabbage     0.143
Beans       0.122
Bananas     0.117
Name: price_kes_per_kg, dtype: float64


## 2 · `transform` and `filter`: keep the table's shape

`agg` shrinks each group to one row. Sometimes you want to keep *every row* but add group-level context. That's `transform`.

In [10]:
p = prices.copy()
p["crop_market_mean"] = p.groupby(["crop", "market"])["price_kes_per_kg"].transform("mean")
p["vs_typical_%"] = (p["price_kes_per_kg"] / p["crop_market_mean"] - 1) * 100

# z-score *within* each series: how unusual is this week for THIS crop in THIS market?
p["z"] = p.groupby(["crop", "market"])["price_kes_per_kg"].transform(lambda s: (s - s.mean()) / s.std())
p.nlargest(3, "z")[["date", "crop", "market", "price_kes_per_kg", "crop_market_mean", "z"]].round({"price_kes_per_kg": 2, "crop_market_mean": 2, "z": 2})

,date,crop,market,price_kes_per_kg,crop_market_mean,z
12136,2024-03-25,Cabbage,Machakos Market,46.6,33.23,3.82
12064,2024-03-18,Cabbage,Machakos Market,46.2,33.23,3.70
14555,2024-11-18,Onions,Eldoret Municipal Market,110.8,76.83,3.55


`filter` keeps or drops **whole groups** by a condition; `rank`, `cumsum` and `pct_change` also work within groups; `head(n)` after sorting gives *top n per group*:

In [11]:
busy = prices.groupby("market").filter(lambda g: g["volume_kg"].mean() > 25_000)
print("Markets whose average weekly volume > 25 t:", sorted(busy["market"].unique()))

top2 = (prices.sort_values("price_kes_per_kg", ascending=False)
        .groupby("crop").head(2)[["crop", "market", "date", "price_kes_per_kg"]].sort_values("crop"))
print("\nTwo dearest records per crop:")
print(top2.head(6).to_string(index=False))

Markets whose average weekly volume > 25 t: ['Eldoret Municipal Market', 'Kongowea Market', 'Nakuru Municipal Market', 'Wakulima Market']

Two dearest records per crop:
   crop                market       date  price_kes_per_kg
Bananas       Machakos Market 2024-09-30              69.2
Bananas Meru Municipal Market 2024-04-15              67.8
  Beans       Machakos Market 2024-04-22             213.7
  Beans       Machakos Market 2024-04-29             212.8
Cabbage       Machakos Market 2024-03-25              46.6
Cabbage       Machakos Market 2024-03-18              46.2


> 🤖 **Why machine learning cares.** `transform` is how you build **group-aware features**: 'price relative to this market's normal', 'rank within crop'. Features like these often beat raw columns in ML models.

## 3 · Reshaping: `pivot_table`, `crosstab`, `melt`

A **pivot table** puts one grouping on the rows and another on the columns: a compact two-way summary you can *read* (and later plot as a heatmap).

In [12]:
pt = prices.pivot_table(index="crop", columns="year", values="price_kes_per_kg", aggfunc="mean").round(1)
pt

year,2021,2022,2023,2024
crop,,,,
Bananas,44.5,47.6,50.5,55.1
Beans,137.1,154.0,162.9,166.9
Cabbage,28.3,30.8,31.3,34.5
Kale,34.9,38.4,38.9,42.6
Maize,52.2,67.3,69.4,64.6
Onions,77.1,83.7,88.4,94.7
Potatoes,41.8,47.0,48.9,50.7
Tomatoes,69.1,76.0,78.6,85.3


In [13]:
prices.pivot_table(index="county", columns="crop", values="price_kes_per_kg", aggfunc="mean", margins=True).round(0)

crop,Bananas,Beans,Cabbage,Kale,Maize,Onions,Potatoes,Tomatoes,All
county,,,,,,,,,
Kiambu,NaN,164.0,34.0,41.0,NaN,94.0,50.0,86.0,78.0
Kisumu,49.0,148.0,29.0,38.0,61.0,84.0,45.0,77.0,67.0
Machakos,55.0,166.0,33.0,41.0,69.0,NaN,50.0,NaN,69.0
Meru,53.0,165.0,NaN,41.0,67.0,NaN,49.0,85.0,77.0
Mombasa,52.0,162.0,33.0,41.0,67.0,90.0,49.0,80.0,72.0
Nairobi,50.0,161.0,33.0,40.0,66.0,91.0,49.0,75.0,72.0
Nakuru,47.0,145.0,29.0,35.0,60.0,81.0,44.0,72.0,64.0
Nyeri,45.0,142.0,29.0,35.0,59.0,80.0,44.0,72.0,63.0
Uasin Gishu,44.0,138.0,28.0,34.0,55.0,77.0,41.0,71.0,61.0


`crosstab` counts combinations (frequency tables):

In [14]:
pd.crosstab(prices["market"], prices["crop"])

crop,Bananas,Beans,Cabbage,Kale,Maize,Onions,Potatoes,Tomatoes
market,,,,,,,,
City Market,0,209,209,209,209,209,209,0
Eldoret Municipal Market,209,209,209,209,209,209,209,209
Karatina Market,209,209,209,209,209,209,209,209
Kibuye Market,209,209,209,209,209,209,209,209
Kongowea Market,209,209,209,209,209,209,209,209
Machakos Market,209,209,209,209,209,0,209,0
Meru Municipal Market,209,209,0,209,209,0,209,209
Nakuru Municipal Market,209,209,209,209,209,209,209,209
Thika Market,0,209,209,209,0,209,209,209


> 🧠 **Pause and predict.** Some cells are 0. What does that tell you? (That market doesn't trade that crop; the retail markets only sell 6 of the 8.) Zero-count cells are *information*.

`melt` does the reverse of a pivot: wide → long. Plotting libraries (Seaborn) *love* long format.

In [15]:
long = pt.reset_index().melt(id_vars="crop", var_name="year", value_name="avg_price")
long.head(6)

,crop,year,avg_price
0,Bananas,2021,44.5
1,Beans,2021,137.1
2,Cabbage,2021,28.3
3,Kale,2021,34.9
4,Maize,2021,52.2
5,Onions,2021,77.1


## 4 · Combining tables

Real analysis needs several sources. Two verbs:

* **`pd.concat`**: *stack* tables (same columns, more rows) or glue side by side.
* **`pd.merge`**: *join* tables on a key column (like a database or Excel's VLOOKUP, but much better).

In [16]:
q1 = prices[prices["date"] < "2021-04-01"]
q2 = prices[(prices["date"] >= "2021-04-01") & (prices["date"] < "2021-07-01")]
stacked = pd.concat([q1, q2], ignore_index=True)
print(len(q1), "+", len(q2), "=", len(stacked))

936

 + 936 = 1872


### 4.1 The four joins

```
  left table          right table
  ┌────────┐          ┌────────┐
  │ A      │          │ A      │     inner : only keys in BOTH
  │ B ─────┼── B ─────┼ B      │     left  : all left keys (+ matches)
  │ C ─────┼── C ─────┼ C      │     right : all right keys (+ matches)
  │ D      │          │ E      │     outer : every key from either
  └────────┘          └────────┘
```

In [17]:
left = pd.DataFrame({"county": ["Nairobi", "Nakuru", "Kisumu"], "avg_price": [70, 55, 58]})
right = pd.DataFrame({"county": ["Nairobi", "Nakuru", "Mombasa"], "region": ["Nairobi", "Rift Valley", "Coast"]})
for how in ["inner", "left", "right", "outer"]:
    print(f"--- how='{how}' ---")
    print(left.merge(right, on="county", how=how).to_string(index=False))

--- how='inner' ---
 county  avg_price      region
Nairobi         70     Nairobi
 Nakuru         55 Rift Valley
--- how='left' ---
 county  avg_price      region
Nairobi         70     Nairobi
 Nakuru         55 Rift Valley
 Kisumu         58         NaN
--- how='right' ---
 county  avg_price      region
Nairobi       70.0     Nairobi
 Nakuru       55.0 Rift Valley
Mombasa        NaN       Coast
--- how='outer' ---
 county  avg_price      region
 Kisumu       58.0         NaN
Mombasa        NaN       Coast
Nairobi       70.0     Nairobi
 Nakuru       55.0 Rift Valley


> 💡 **Intuition.** **Default to `how='left'`** when you're *enriching* your main table with extra columns: you keep every original row, and gaps (NaN) honestly show what failed to match.

### 4.2 Enrich the price table with county facts

In [18]:
enriched = prices.merge(counties, on="county", how="left", validate="many_to_one")
print(prices.shape, "->", enriched.shape)
enriched[["date", "crop", "county", "region", "population_2019", "price_kes_per_kg"]].head(3)

(15048, 11) -> (15048, 13)


,date,crop,county,region,population_2019,price_kes_per_kg
0,2021-01-04,Beans,Nairobi,Nairobi,4397073,143.5
1,2021-01-04,Cabbage,Nairobi,Nairobi,4397073,31.1
2,2021-01-04,Kale,Nairobi,Nairobi,4397073,39.1


`validate="many_to_one"` says: *"many price rows per county, but each county must appear **once** in `counties`."* If that's false, pandas raises an error instead of quietly duplicating rows.

### 4.3 Danger zone ⚠️: the three silent merge killers

> ⚠️ **Common mistake.** **1. Row explosion.** Joining on a key that is *not unique* on both sides multiplies rows. **2. Key mismatch.** `'Uasin Gishu'` ≠ `'Uasin-Gishu'`, so rows silently fail to match. **3. Wrong join type.** An `inner` join quietly drops unmatched rows. Defence: *always compare row counts before and after* and use `validate=` / `indicator=True`.

**Killer 1: row explosion.** Nairobi has *two* markets. Joining a county-level production table to the market list on `county` double-counts Nairobi:

In [19]:
prod23 = production[production["year"] == 2023]
bad = prod23.merge(markets[["market", "county"]], on="county", how="left")
print("production rows:", len(prod23), "-> after merge:", len(bad), " (rows exploded: Nairobi has 2 markets)")
try:
    prod23.merge(markets[["market", "county"]], on="county", validate="many_to_one")
except Exception as e:
    print("\nvalidate caught it ->", type(e).__name__)

production rows: 51 -> after merge: 53  (rows exploded: Nairobi has 2 markets)

validate caught it -> MergeError


**Killer 2: key mismatch.** Let's join production to an *annual price summary*. `indicator=True` adds a `_merge` column telling us where each row came from:

In [20]:
annual_price = (prices.groupby(["county", "crop", "year"], as_index=False)["price_kes_per_kg"].mean()
                .rename(columns={"price_kes_per_kg": "avg_price"}))
j = production.merge(annual_price, on=["county", "crop", "year"], how="left", indicator=True)
print(j["_merge"].value_counts())
print("\nProduction rows with no price match, by county:")
print(j[j["_merge"] == "left_only"]["county"].value_counts().to_string())

_merge
both          176
left_only      28
right_only      0
Name: count, dtype: int64

Production rows with no price match, by county:
county
Meru           8
Kiambu         8
Machakos       8
Uasin-Gishu    4


Two different causes are hiding in there: **(a)** `Uasin-Gishu` is spelt differently in the Excel file, and **(b)** some counties grow a crop that our markets simply don't report.
Cause (a) we can fix; cause (b) is a genuine gap. Diagnose first, *then* repair:

In [21]:
production["county"] = production["county"].str.replace("-", " ", regex=False)
j = production.merge(annual_price, on=["county", "crop", "year"], how="left", indicator=True)
print(j["_merge"].value_counts().to_string())
print("\nStill unmatched (true gaps), by county:", j[j["_merge"] == "left_only"]["county"].value_counts().to_dict())

_merge
both          180
left_only      24
right_only      0

Still unmatched (true gaps), by county: {'Meru': 8, 'Kiambu': 8, 'Machakos': 8}


> 🌍 **In the real world.** Merge problems rarely announce themselves. A shape that changes from 15,048 to 15,100 rows, or a column that is suddenly 20% NaN, is your only clue. Make it a habit: **print shapes and `_merge` counts after every merge.**

### 4.4 Combining prices with weather

The prices are weekly; the weather is monthly. To merge them we first bring the prices to the **same grain** (county × year × month), then join on all three keys.

In [22]:
monthly = (prices.groupby(["county", "year", "month"], as_index=False)
           .agg(avg_price=("price_kes_per_kg", "mean"), volume_t=("volume_kg", lambda s: s.sum() / 1000)))
panel = monthly.merge(weather, on=["county", "year", "month"], how="left", validate="one_to_one")
print(monthly.shape, "->", panel.shape, "| missing rainfall:", panel["rainfall_mm"].isna().sum())
panel.head(3)

(432, 5) -> (432, 7) | missing rainfall: 0


,county,year,month,avg_price,volume_t,rainfall_mm,avg_temp_c
0,Kiambu,2021,1,71.620833,398.77,72.2,18.9
1,Kiambu,2021,2,72.812500,410.44,73.8,19.4
2,Kiambu,2021,3,74.460000,481.00,103.8,19.9


> 💡 **Intuition.** Joining tables at the wrong *grain* (weekly vs monthly) is the most common source of nonsense results. Before merging, ask: **what does one row mean in each table?**

---
# 🧑‍🏫 Worked examples

## Worked example 1: "Where is maize cheapest?"

> **Thinking steps.** (1) Filter to maize. (2) Group by county. (3) Mean of price. (4) Sort. (5) Express the gap against the cheapest county: a number a manager can use.

In [23]:
mz = prices[prices["crop"] == "Maize"].groupby("county")["price_kes_per_kg"].mean().sort_values()
gap = ((mz / mz.iloc[0] - 1) * 100).round(1)
print(pd.DataFrame({"avg_price": mz.round(1), "% above cheapest": gap}))

             avg_price  % above cheapest
county                                  
Uasin Gishu       55.4               0.0
Nyeri             58.6               5.8
Nakuru            59.9               8.0
Kisumu            61.4              10.7
Nairobi           66.2              19.4
Mombasa           66.8              20.5
Meru              67.4              21.7
Machakos          68.6              23.8


## Worked example 2: "Do counties that grow more maize have cheaper maize?"

> **Thinking steps.** (1) Average 2023 maize price per county. (2) 2023 maize production per county from the Excel table. (3) Merge on `county` (inner (we need both). (4) Check row counts. (5) Correlate) and *think* before concluding.

In [24]:
mp = (prices[(prices["crop"] == "Maize") & (prices["year"] == 2023)]
      .groupby("county", as_index=False)["price_kes_per_kg"].mean())
pr = production[(production["crop"] == "Maize") & (production["year"] == 2023)][["county", "production_tonnes"]]
jm = mp.merge(pr, on="county", how="inner", validate="one_to_one")
print("counties with price:", len(mp), "| with production:", len(pr), "| matched:", len(jm))
print(jm.sort_values("production_tonnes", ascending=False).round(1).to_string(index=False))
print("\ncorrelation(production, price) =", round(jm["production_tonnes"].corr(jm["price_kes_per_kg"]), 2))

counties with price: 8 | with production: 8 | matched: 7
     county  price_kes_per_kg  production_tonnes
Uasin Gishu              60.7           262956.1
     Nakuru              65.0           191366.1
       Meru              72.5           152128.5
   Machakos              76.1           136111.8
     Kisumu              67.1           100993.2
      Nyeri              64.0            42054.1
    Mombasa              74.6             1998.0

correlation(production, price) = -0.47


> 🧠 **Pause and predict.** A moderately negative correlation: big producers tend to have cheaper maize. Can we say *producing more causes lower prices*? Not yet, with only 7 counties the number is fragile, and production isn't the whole story either: the dearest maize is in *retail* markets (Machakos, Meru, City Market), which carry a retail premium regardless of local production. Correlation shows a pattern; it does not explain it. We'll practise exactly this reasoning in Lesson 8.

## Worked example 3: "How did the drought hit each crop?"

> **Thinking steps.** (1) Yearly average price per crop (pivot). (2) 2022→2023 and 2021→2023 % change. (3) Rank. The ones that jumped most are drought-sensitive.

In [25]:
yp = prices.pivot_table(index="crop", columns="year", values="price_kes_per_kg", aggfunc="mean")
res = pd.DataFrame({"2021→2023 %": ((yp[2023] / yp[2021] - 1) * 100).round(1)}).sort_values("2021→2023 %", ascending=False)
res

,2021→2023 %
crop,
Maize,32.8
Beans,18.9
Potatoes,17.2
Onions,14.7
Tomatoes,13.7
Bananas,13.5
Kale,11.5
Cabbage,10.5


---
# 🏋️ Problems

Variables from Setup: `prices` (with `year`, `month`), `production`, `counties`, `weather`, `markets`.
Note: `production` still has the `Uasin-Gishu` spelling issue unless *you* fix it in your own code.

### 🧩 Problem 5.1: Mean price by crop

Compute the **mean price per crop** and return it as a dictionary `{crop: mean}` with values rounded to 2 decimals.

*Expected type:* dict.

<details>
<summary>💡 Hint 1</summary>

`groupby('crop')['price_kes_per_kg'].mean()` gives a Series indexed by crop.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Round it, then `.to_dict()`.

</details>

*Auto-check id:* `05-p1`  ·  *Set the variable `answer`, then run the cell.*

In [26]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p1', answer)

⏳ 05-p1: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.2: Tomato capital

Which **county** has the highest **mean tomato price**?

*Expected type:* a string.

<details>
<summary>💡 Hint 1</summary>

Filter tomatoes, group by county, average the price.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`.idxmax()` returns the label (the county) of the largest value.

</details>

*Auto-check id:* `05-p2`  ·  *Set the variable `answer`, then run the cell.*

In [27]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p2', answer)

⏳ 05-p2: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.3: Maize in 2022 (pivot)

Build a pivot table of **mean price** with crops as rows and years as columns. Return the cell for **Maize, 2022**, rounded to 2 decimals.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

`pivot_table(index='crop', columns='year', values='price_kes_per_kg', aggfunc='mean')`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Read one cell with `.loc['Maize', 2022]` (the column label is an integer).

</details>

*Auto-check id:* `05-p3`  ·  *Set the variable `answer`, then run the cell.*

In [28]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p3', answer)

⏳ 05-p3: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.4: The biggest spike (transform)

Within **each (crop, market) series**, standardise the price: `(price − series mean) / series std` (pandas default `std`). Return the **largest** standardised value in the whole table, rounded to 2 decimals.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

`groupby(['crop', 'market'])['price_kes_per_kg'].transform(...)` keeps the original row count.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Inside, use `lambda s: (s - s.mean()) / s.std()`. Then `.max()`.

</details>

*Auto-check id:* `05-p4`  ·  *Set the variable `answer`, then run the cell.*

In [29]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p4', answer)

⏳ 05-p4: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.5: Most volatile crop

Using the **pooled** price of each crop (all markets, all weeks), compute the coefficient of variation (`std / mean`). Which crop has the **highest** CV?

*Expected type:* a string.

<details>
<summary>💡 Hint 1</summary>

`g = prices.groupby('crop')['price_kes_per_kg']`, then `g.std() / g.mean()`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`.idxmax()` on the result.

</details>

*Auto-check id:* `05-p5`  ·  *Set the variable `answer`, then run the cell.*

In [30]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p5', answer)

⏳ 05-p5: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.6: Top maize region in 2023 (merge)

Merge `production` with `counties` (left join on `county`). Among **Maize** rows in **2023**, which **region** had the highest total `production_tonnes`?

*Expected type:* a string.

<details>
<summary>💡 Hint 1</summary>

`production.merge(counties, on='county', how='left')` adds the `region` column.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Filter crop and year, `groupby('region')['production_tonnes'].sum()`, then `idxmax()`.

</details>

*Auto-check id:* `05-p6`  ·  *Set the variable `answer`, then run the cell.*

In [31]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p6', answer)

⏳ 05-p6: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.7: Orphan rows

How many rows of `production` (as loaded in Setup, *unfixed*) have a `county` that does **not** exist in the `counties` table?

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

`Series.isin(other_series)` tests membership; `~` negates it.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Or merge with `indicator=True` and count `'left_only'`.

</details>

*Auto-check id:* `05-p7`  ·  *Set the variable `answer`, then run the cell.*

In [32]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p7', answer)

⏳ 05-p7: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.8: Does rain move tomato prices?

Compute the **monthly mean tomato price per county** (group by `county, year, month`). Merge it with `weather` on `['county','year','month']` (inner join). Return the **Pearson correlation** between the mean price and `rainfall_mm`, rounded to 3 decimals.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

Filter tomatoes, `groupby(['county','year','month'], as_index=False)['price_kes_per_kg'].mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`merged = monthly.merge(weather, on=[...], how='inner')`.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`merged['price_kes_per_kg'].corr(merged['rainfall_mm'])`.

</details>

*Auto-check id:* `05-p8`  ·  *Set the variable `answer`, then run the cell.*

In [33]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p8', answer)

⏳ 05-p8: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.9: Busy series

A *series* is one (crop, market) pair. How many series have an **average weekly volume above 25,000 kg**?

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

Group by both keys and average `volume_kg`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Compare the resulting Series to 25000 and count the Trues.

</details>

*Auto-check id:* `05-p9`  ·  *Set the variable `answer`, then run the cell.*

In [34]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p9', answer)

⏳ 05-p9: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 5.10: Market league table (named aggregation)

Build a summary by `market` with named aggregation: `total_volume_t` = total `volume_kg` **in tonnes** (÷1000). Return the total volume in tonnes for **Wakulima Market**, rounded to 1 decimal.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

`prices.groupby('market').agg(total_volume_t=('volume_kg', lambda s: s.sum() / 1000))`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Select the row with `.loc['Wakulima Market', 'total_volume_t']`.

</details>

*Auto-check id:* `05-p10`  ·  *Set the variable `answer`, then run the cell.*

In [35]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('05-p10', answer)

⏳ 05-p10: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

## ✅ Check your understanding

Answer in your head (or out loud!) first, *then* open the answer.

**Q1. What's the difference between `groupby().agg()` and `groupby().transform()`?**

<details>
<summary>Show answer</summary>

`agg` returns **one row per group**; `transform` returns a result **the same length as the original** (each row gets its group's value), so it can be added back as a column.

</details>

**Q2. You merge a 15,000-row table with a lookup table and get 15,900 rows. What happened and how do you prevent it?**

<details>
<summary>Show answer</summary>

The key is **not unique in the lookup table**, so matching rows were duplicated. Check uniqueness first or use `validate='many_to_one'`, which raises an error instead.

</details>

**Q3. Why default to a `left` join when adding extra columns to your main table?**

<details>
<summary>Show answer</summary>

It keeps *every* original row; unmatched ones get NaN: visible evidence of mismatches, instead of rows silently vanishing as in an `inner` join.

</details>

**Q4. When is `pivot_table` better than `groupby`?**

<details>
<summary>Show answer</summary>

When you want a *two-way* layout (rows = one grouping, columns = another) that's easy to read or plot as a heatmap. `groupby` with two keys gives the same numbers in a long, stacked shape.

</details>

**Q5. Why compare CV (std/mean) instead of std when judging volatility across crops?**

<details>
<summary>Show answer</summary>

Std is in the crop's own units, so expensive crops look more volatile just because their prices are bigger. CV is *relative* variability, comparable across crops.

</details>

## 🧾 Cheat-sheet

| I want to… | Code |
|---|---|
| Mean per group | `df.groupby("g")["x"].mean()` |
| Many stats | `.agg(["mean","std"])` or named: `.agg(avg=("x","mean"), n=("id","count"))` |
| Keep keys as columns | `groupby(..., as_index=False)` or `.reset_index()` |
| Add group stat to every row | `df.groupby("g")["x"].transform("mean")` |
| Keep only some groups | `df.groupby("g").filter(lambda d: len(d) > 10)` |
| Top n per group | `df.sort_values("x", ascending=False).groupby("g").head(2)` |
| Pivot / counts / long format | `df.pivot_table(index=, columns=, values=, aggfunc=)`, `pd.crosstab(a, b)`, `df.melt(id_vars=)` |
| Stack tables | `pd.concat([a, b], ignore_index=True)` |
| Join | `a.merge(b, on="k", how="left", validate="many_to_one", indicator=True)` |
| Join on several keys | `on=["county", "year", "month"]` |

## ➡️ Next up
**Lesson 6; Matplotlib.** Numbers tell you *what*; pictures tell you *why*. Time to draw.